Fixed UDF module errors by replacing modin .apply() with a JavaScript UDF and pure SQL CTAS
*Co-authored with CoCo*

In [ ]:
#!pip install -r requirements.txt

In [ ]:
import modin.pandas as pd
import numpy as np
import datetime
import snowflake.snowpark.modin.plugin
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

In [ ]:
# Import python packages
import logging

# set up the logger
logger_name = 'process_logger'
logger = logging.getLogger(logger_name)
logger.setLevel(logging.INFO)

In [ ]:
%run ./PY_INITIALIZE.ipynb

In [ ]:

# get snowpark session
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Add packages needed for UDF execution in .apply() calls
session.add_packages('snowflake-snowpark-python', 'pandas', 'cloudpickle', 'modin')

# set the default database and schema for the session
session.use_schema(f"{database_name}.{schema_name}")

In [ ]:
clmspt = pd.read_snowflake(f"{database_name}.{schema_name}.CLMPT")

logging.info(f"DataFrame loaded successfully with {len(clmspt)} rows and {len(clmspt.columns)} columns")
#clmspt.head()

In [ ]:
today = datetime.date.today()
year = today.year
# 1. Ensure your date column is in a datetime format
clmspt["PT_DOB"] = clmspt["PT_DOB"].astype(str)
#clmspt["PT_DOB"] = clmspt["PT_DOB"].str.strip('"')
clmspt["PT_DOB"] = pd.to_datetime(clmspt["PT_DOB"])

# 2. Extract the year and store it in a new column 'brth_yr'
clmspt['brth_yr'] = clmspt["PT_DOB"].dt.year
clmspt['PT_AGE']=year - clmspt['brth_yr']
clmspt = clmspt.drop(['brth_yr'],axis=1)
clmspt['PT_ZIP'] = clmspt['PT_ZIP'].astype(int)

In [ ]:
import math

def convert_chars(text):
    if text is None:
        return '0'
    if isinstance(text, float):
        if math.isnan(text):
            return '0'
        return float(text)
    if isinstance(text, int):
        return float(text)
    text = str(text)
    if text == '' or text == 'None' or text == 'nan':
        return '0'
    result = ""
    for char in text:
        if char.isdigit():
            result = result + char
        else:
            result = result + str(ord(char))
    return result

In [ ]:
clmspt["SUBCD_NBR"] = clmspt["CPT_CD"].apply(
    lambda x: convert_chars(x)
)

In [ ]:
clmspt["ICDCD_NUMCODED"] = clmspt["ICD_CD"].apply(
    lambda x: convert_chars(x)
)

clmspt["ICDCD_NUMCODED"] = clmspt["ICD_CD"].apply(
lambda x: pd.session.sql(f"SELECT process_text_js('{x}')").collect()[0][0] 
    if x else ""
)

In [ ]:
clmspt['OMC_ICD_RISK_NBR'].fillna(0, inplace=True)

In [ ]:
clmspt['ICDCD_NUMCODED'] = clmspt['ICDCD_NUMCODED'].astype(float)
clmspt['SUBCD_NBR'] = clmspt['SUBCD_NBR'].astype(float)

clmspt['OMC_ICD_RISK_NBR'] = clmspt['OMC_ICD_RISK_NBR'].astype(str).str.replace('"', "", regex=False) 
clmspt['OMC_ICD_RISK_NBR'] = clmspt['OMC_ICD_RISK_NBR'].astype(int)
clmspt['PT_AGE'] = clmspt['PT_AGE'].astype(int)
clmspt['PT_ZIP'] = clmspt['PT_ZIP'].astype(str).str.replace('"', "", regex=False) 
clmspt['PT_ZIP'] = clmspt['PT_ZIP'].astype(int)
print(clmspt['PT_ZIP'][:10])

In [ ]:
clmspt.drop(columns=['PT_DOB'], inplace=True)

In [ ]:
clmspt.to_snowflake(f"{database_name}.{schema_name}.Final_Table", if_exists='replace', index=False)